# Exploration des données

Objectifs : distribution des classes (déséquilibre attendu), co-occurrences, exemples visuels, qualité des
labels, taille des images, nombre de patches par image source. **Note tes constats en bas du notebook :
ils serviront dans le README et en entretien.**

Prérequis : `data/processed/index.csv` (et `splits.csv`), produits par `chantier_defects.data.index` / `.split`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from chantier_defects.data.split import class_columns
from chantier_defects.labels import label_fr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/processed/index.csv")
classes = class_columns(df)
print(f"{len(df)} images, {df['group'].nunique()} images sources, classes : {classes}")
df.head()

## Distribution des classes

In [ ]:
counts = df[classes].sum().rename(index=label_fr)
counts["fond (aucun défaut)"] = int((df[classes].sum(axis=1) == 0).sum())
ax = counts.sort_values().plot.barh(figsize=(7, 3), title="Nombre d'images par classe")
for i, v in enumerate(counts.sort_values()):
    ax.text(v, i, f" {v} ({v / len(df):.1%})", va="center")
plt.show()
print("Nombre de défauts par image :", df[classes].sum(axis=1).value_counts().sort_index().to_dict())

## Co-occurrences (multi-label)

In [ ]:
if len(classes) > 1:
    co = df[classes].T.dot(df[classes])
    co.index = co.columns = [label_fr(c) for c in classes]
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(co, cmap="Blues")
    ax.set_xticks(range(len(co)), co.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(co)), co.index)
    for i in range(len(co)):
        for j in range(len(co)):
            ax.text(j, i, int(co.iloc[i, j]), ha="center", va="center", fontsize=8)
    ax.set_title("Co-occurrences des défauts")
    plt.colorbar(im)
    plt.show()

## Exemples par classe

Regarde aussi les labels douteux : fissure invisible, ombre ou joint étiqueté comme défaut, etc.

In [ ]:
rng = np.random.default_rng(0)
groups = {label_fr(c): df[df[c] == 1] for c in classes} | {"fond": df[df[classes].sum(axis=1) == 0]}
n = 6
fig, axes = plt.subplots(len(groups), n, figsize=(n * 2, len(groups) * 2.1), squeeze=False)
for row, (name, sub) in enumerate(groups.items()):
    sample = sub.sample(min(n, len(sub)), random_state=0)
    for col, ax in enumerate(axes[row]):
        ax.axis("off")
        if col < len(sample):
            ax.imshow(Image.open(sample.iloc[col]["path"]).convert("RGB"))
        if col == 0:
            ax.set_title(name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

## Tailles d'images et patches par image source

In [ ]:
sizes = pd.DataFrame(
    [Image.open(p).size for p in df["path"].sample(min(500, len(df)), random_state=0)], columns=["w", "h"]
)
print(sizes.describe().round(0))
per_group = df.groupby("group").size()
print(f"Patches par image source : médiane {per_group.median():.0f}, max {per_group.max()}")
per_group.plot.hist(bins=30, figsize=(6, 2.5), title="Patches par image source")
plt.show()

## Vérification du split (si `splits.csv` existe)

In [ ]:
splits_path = ROOT / "data/processed/splits.csv"
if splits_path.exists():
    sp = pd.read_csv(splits_path)
    summary = sp.groupby("split")[classes].mean().rename(columns=label_fr).round(3)
    summary.insert(0, "n", sp["split"].value_counts())
    display(summary)
    assert sp.groupby("group")["split"].nunique().max() == 1, "fuite : un groupe dans plusieurs splits"
    print("Aucun groupe partagé entre splits.")

## Constats (à rédiger)

- Déséquilibre : …
- Classes rares et conséquences sur les métriques (rappel par classe, PR-AUC plutôt que l'accuracy) : …
- Qualité des labels (exemples douteux repérés) : …
- Confusions visuelles probables (ombres, joints, taches, bullage) : …
- Risque de fuite entre patches voisins et choix du split groupé : …